# Multivariate DFM: 10+ Series with Missing Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-deane/course-creator/blob/main/courses/dynamic-factor-models/quick-starts/02_multivariate_example.ipynb)

**Goal:** Extract interpretable factors from 10 macroeconomic series and handle missing data gracefully.

**Time:** 15 minutes

**What you'll learn:** How DFM shines with many variables and how it handles missing data better than traditional methods.

---

## Quick Win: Comprehensive Economic Nowcasting

We'll build a "nowcasting" system using 10 key economic indicators.

In [ ]:
# Install if running in Colab
try:
    import google.colab
    !pip install -q pandas-datareader statsmodels
except:
    pass

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from pandas_datareader import data as web
from datetime import datetime

print("✓ Libraries loaded successfully!")

In [ ]:
# Download 10 macroeconomic indicators from FRED
start = datetime(2010, 1, 1)
end = datetime(2023, 12, 31)

indicators = {
    # Output & Income
    'INDPRO': 'Industrial Production',
    'PAYEMS': 'Total Employment',
    'PI': 'Personal Income',
    
    # Consumption & Sales
    'PCE': 'Personal Consumption',
    'RETAILSMSA': 'Retail Sales',
    
    # Investment & Housing
    'HOUST': 'Housing Starts',
    'GPDI': 'Private Investment',
    
    # Labor Market
    'UNRATE': 'Unemployment Rate',
    'ICSA': 'Initial Claims',
    
    # Prices
    'CPIAUCSL': 'Consumer Price Index',
}

print("📊 Downloading 10 economic indicators from FRED...")
data_raw = pd.DataFrame()
for code, name in indicators.items():
    try:
        series = web.DataReader(code, 'fred', start, end)
        data_raw[name] = series.iloc[:, 0]
        print(f"  ✓ {name}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

print(f"\n✓ Downloaded {len(data_raw.columns)} series with {len(data_raw)} observations")
print(f"Date range: {data_raw.index[0].date()} to {data_raw.index[-1].date()}")

## Visualize: The Missing Data Challenge

Real-world economic data often has missing values due to different reporting schedules.

In [ ]:
# Visualize missing data pattern
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Missing data heatmap
missing_mask = data_raw.isnull().astype(int)
sns.heatmap(missing_mask.T, cmap='RdYlGn_r', cbar_kws={'label': 'Missing'}, 
            yticklabels=data_raw.columns, ax=axes[0])
axes[0].set_title('Missing Data Pattern (Red = Missing)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time')

# Plot 2: Missing data percentage
missing_pct = (data_raw.isnull().sum() / len(data_raw) * 100).sort_values(ascending=False)
missing_pct.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Missing Data by Indicator', fontsize=14, fontweight='bold')
axes[1].set_xlabel('% Missing')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\n📊 Missing Data Summary:")
print(f"  Total observations: {len(data_raw) * len(data_raw.columns):,}")
print(f"  Missing values: {data_raw.isnull().sum().sum():,}")
print(f"  Complete cases: {data_raw.dropna().shape[0]:,}")
print(f"\n💡 DFM can work with ALL {len(data_raw)} observations (no need to drop rows)!")

In [ ]:
# Prepare data: Standardize but keep NaNs (DFM handles them)
data_std = (data_raw - data_raw.mean()) / data_raw.std()

# For comparison: how much data would we lose if we dropped NaNs?
data_complete = data_std.dropna()
print(f"With complete-case deletion: {len(data_complete)} obs ({len(data_complete)/len(data_std)*100:.1f}%)")
print(f"With DFM (handles NaNs):     {len(data_std)} obs (100%)")
print(f"\n→ DFM preserves {len(data_std) - len(data_complete)} additional observations!")

In [ ]:
# Fit Dynamic Factor Model (handles missing data automatically)
print("\n🔧 Fitting Dynamic Factor Model with 3 factors...")
print("   (This may take 30-60 seconds with 10 series...)\n")

model = DynamicFactor(
    data_std,
    k_factors=3,        # Extract 3 factors from 10 series
    factor_order=2      # AR(2) dynamics for factors
)

results = model.fit(disp=True, maxiter=500)

print("\n✓ Model fitted successfully!")
print(f"Log-likelihood: {results.llf:.2f}")
print(f"AIC: {results.aic:.2f}")
print(f"BIC: {results.bic:.2f}")

## Visualize: Three Interpretable Economic Factors

In [ ]:
# Extract factors
factors = results.factors.filtered

# Create factor DataFrame
factors_df = pd.DataFrame(
    factors,
    index=data_std.index,
    columns=['Factor 1', 'Factor 2', 'Factor 3']
)

# Plot factors
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Add recession shading (2020 COVID recession)
recession_start = pd.Timestamp('2020-02-01')
recession_end = pd.Timestamp('2020-04-30')

for i, ax in enumerate(axes):
    factor_name = f'Factor {i+1}'
    ax.plot(factors_df.index, factors_df[factor_name], linewidth=2, color=f'C{i}')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.axvspan(recession_start, recession_end, alpha=0.2, color='gray', label='COVID Recession')
    ax.set_title(f'{factor_name}: Overall Economic Activity', fontsize=12, fontweight='bold')
    ax.set_ylabel('Factor Value')
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend()

axes[2].set_xlabel('Date')
plt.tight_layout()
plt.show()

print("\n💡 Notice: Factor 1 shows a sharp drop during COVID recession")

## Factor Interpretation: What Do They Mean?

Factor loadings tell us which indicators each factor represents.

In [ ]:
# Extract factor loadings
loadings = results.params['loading'].values.reshape(len(data_std.columns), 3)
loadings_df = pd.DataFrame(
    loadings,
    index=data_std.columns,
    columns=['Factor 1', 'Factor 2', 'Factor 3']
)

# Visualize loadings
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Loading'}, ax=ax, vmin=-2, vmax=2)
ax.set_title('Factor Loadings: Which Indicators Load on Which Factors?', 
             fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print("\n📊 Factor Interpretation:")
print("\nFactor 1 (high loadings):")
top_f1 = loadings_df['Factor 1'].abs().nlargest(3)
for idx, val in top_f1.items():
    print(f"  → {idx}: {loadings_df.loc[idx, 'Factor 1']:.2f}")

print("\nFactor 2 (high loadings):")
top_f2 = loadings_df['Factor 2'].abs().nlargest(3)
for idx, val in top_f2.items():
    print(f"  → {idx}: {loadings_df.loc[idx, 'Factor 2']:.2f}")

print("\nFactor 3 (high loadings):")
top_f3 = loadings_df['Factor 3'].abs().nlargest(3)
for idx, val in top_f3.items():
    print(f"  → {idx}: {loadings_df.loc[idx, 'Factor 3']:.2f}")

print("\n💡 Interpretation Guide:")
print("  - Large positive loading: variable moves WITH factor")
print("  - Large negative loading: variable moves AGAINST factor")
print("  - Near-zero loading: variable independent of that factor")

## How It Works: Dimensionality Reduction in Action

**The power of DFM:**
```
10 correlated indicators → 3 independent factors
```

**Benefits:**
1. **Simpler interpretation**: 3 factors easier than 10 series
2. **Noise reduction**: Factors filter out idiosyncratic noise
3. **Missing data**: Uses all available information at each time point
4. **Forecasting**: Forecast factors, then map back to original series

In [ ]:
# Demonstrate reconstruction: Original ≈ Factors × Loadings + Error
reconstructed = results.fittedvalues

# Pick one indicator to show reconstruction
indicator = 'Industrial Production'
idx = list(data_std.columns).index(indicator)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(data_std.index, data_std[indicator], label='Original', linewidth=2, alpha=0.7)
ax.plot(reconstructed.index, reconstructed.iloc[:, idx], 
        label='Reconstructed (from 3 factors)', linewidth=2, linestyle='--')
ax.set_title(f'{indicator}: Original vs Reconstructed from Factors', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Standardized Value')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate R-squared
original = data_std[indicator].dropna()
recon = reconstructed.iloc[:, idx].loc[original.index]
r2 = 1 - np.sum((original - recon)**2) / np.sum((original - original.mean())**2)
print(f"\n✓ R² = {r2:.3f}")
print(f"  → 3 factors explain {r2*100:.1f}% of variance in {indicator}")

## Modify This: Choose Optimal Number of Factors

How many factors should you use? Try different values and compare.

In [ ]:
# TODO: Test different numbers of factors
k_values = [1, 2, 3, 4, 5]  # Try adding 6, 7, etc.

results_comparison = []

print("🔧 Testing different numbers of factors...\n")
for k in k_values:
    print(f"  Fitting k={k}...", end=" ")
    try:
        model_k = DynamicFactor(data_std, k_factors=k, factor_order=2)
        result_k = model_k.fit(disp=False, maxiter=500)
        results_comparison.append({
            'k_factors': k,
            'AIC': result_k.aic,
            'BIC': result_k.bic,
            'LogLik': result_k.llf
        })
        print(f"✓ AIC={result_k.aic:.1f}, BIC={result_k.bic:.1f}")
    except Exception as e:
        print(f"✗ Failed: {e}")

# Plot comparison
comparison_df = pd.DataFrame(results_comparison)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot AIC
axes[0].plot(comparison_df['k_factors'], comparison_df['AIC'], 
             marker='o', linewidth=2, markersize=8, color='steelblue')
best_aic = comparison_df.loc[comparison_df['AIC'].idxmin(), 'k_factors']
axes[0].axvline(best_aic, color='red', linestyle='--', label=f'Best k={int(best_aic)}')
axes[0].set_title('AIC vs Number of Factors (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Factors')
axes[0].set_ylabel('AIC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot BIC
axes[1].plot(comparison_df['k_factors'], comparison_df['BIC'], 
             marker='s', linewidth=2, markersize=8, color='coral')
best_bic = comparison_df.loc[comparison_df['BIC'].idxmin(), 'k_factors']
axes[1].axvline(best_bic, color='red', linestyle='--', label=f'Best k={int(best_bic)}')
axes[1].set_title('BIC vs Number of Factors (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Factors')
axes[1].set_ylabel('BIC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Model Selection:")
print(f"  AIC suggests: k = {int(best_aic)} factors")
print(f"  BIC suggests: k = {int(best_bic)} factors (more conservative)")
print(f"\n💡 BIC penalizes complexity more → often prefers fewer factors")

## Copy-Paste Template: Production-Ready Multivariate DFM

In [ ]:
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
import pandas as pd
import numpy as np

def fit_dfm_pipeline(data, k_factors=3, factor_order=2):
    """
    Complete DFM pipeline: fit, extract factors, get loadings.
    
    Parameters:
    - data: DataFrame with time series (can have NaNs)
    - k_factors: Number of factors to extract
    - factor_order: AR order for factor dynamics
    
    Returns:
    - results: Fitted model results
    - factors_df: DataFrame of estimated factors
    - loadings_df: DataFrame of factor loadings
    """
    # Standardize
    data_std = (data - data.mean()) / data.std()
    
    # Fit model
    model = DynamicFactor(data_std, k_factors=k_factors, factor_order=factor_order)
    results = model.fit(disp=False, maxiter=500)
    
    # Extract factors
    factors = results.factors.filtered
    factors_df = pd.DataFrame(
        factors,
        index=data.index,
        columns=[f'Factor_{i+1}' for i in range(k_factors)]
    )
    
    # Extract loadings
    loadings = results.params['loading'].values.reshape(len(data.columns), k_factors)
    loadings_df = pd.DataFrame(
        loadings,
        index=data.columns,
        columns=[f'Factor_{i+1}' for i in range(k_factors)]
    )
    
    return results, factors_df, loadings_df

# USAGE:
# results, factors, loadings = fit_dfm_pipeline(your_data, k_factors=3)
# print(f"AIC: {results.aic:.2f}")
# factors.plot(figsize=(12, 6))

## What's Next?

You just handled 10 time series with missing data and extracted interpretable factors! Here's where to go:

1. **`../templates/dfm_nowcasting_template.py`** - Production nowcasting system
2. **`../concepts/visual_guides/factor_rotation.md`** - Make factors more interpretable
3. **`../modules/module_03_applications/notebooks/01_gdp_nowcasting.ipynb`** - Real-time GDP estimation
4. **`../recipes/handle_mixed_frequencies.py`** - Combine monthly + quarterly data

---

**Key Takeaways:**
- DFM reduces dimensionality: 10 series → 3 factors
- Handles missing data automatically (no imputation needed)
- Factor loadings reveal economic structure
- Use AIC/BIC to choose optimal number of factors
- Factors are often interpretable (e.g., "real activity", "inflation", "financial conditions")